# 1.14 如何接著寫下去？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：下一字對齊：問題與答案錯開一格**

同一格 logits 預測下一個字；shift 只做一次。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/shift.svg")))

**先想一想：**固定 seed 後重跑，能得到同一段文字嗎？

每抽出一個字，就把它當成下一次的前文。生成是把同一個「猜下一字」問題重複使用，不是一次吐出整篇文章。

**把直覺寫成數學：**bigram 生成 $s_{t+1}\sim p(\cdot\mid s_t)$；這裡用計數分佈，不需要訓練權重。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
chars = ["貓", "看", "狗", "。"]
p = torch.tensor([[0.05, 0.85, 0.05, 0.05], [0.4, 0.05, 0.5, 0.05], [0.05, 0.7, 0.05, 0.2], [0.5, 0.05, 0.4, 0.05]])
g = torch.Generator().manual_seed(42)
current = 0
output = [chars[current]]
for _ in range(12):
    current = torch.multinomial(p[current], 1, generator=g).item()
    output.append(chars[current])
print("".join(output))

**如何讀結果：**隨機抽樣可以有不常見結果；短字串通順不代表模型理解世界。

**只改一件事：**只改 seed，保持 p 不變。
